# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression.

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept).
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already.
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [ ]:
# NAME(s) = "Macy Moore, Ava Lawless, Nathaniel Horton"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

**Cost function to use**:

Binary: Binary cross-entropy Loss

Multi-class: Categorical cross-entropy Loss

Regression: MSE (Mean Squared Error)


**Changes needed in output layer**:

Binary classification needs to have a single unit with a sigmoid activation to turn the final output into a probability between 0 and 1, allowing us to set a threshold for classifying between positive and negative class (likely 0.5).

Multi-class classification needs one unit per class (3) and use a softmax activation, turning the outputs into probabilities that sum up to 1. The predicted class is whichever class has the highest outputted probability.

Regression needs one unit and no activation function since it stays linear, and the model can just predict a real number rather than being limited to a specific range.

**Why it's important to standardize data before training**

It's important to standardize data before training because features can be on different scales, and when dealing with numeric features and methods such as gradient descent, larger scales can have much larger effects on the output for updates. Standardizing the data solves this issue as it puts the features on the same scale and training can converge quicker without worrying about larger scale features dominating the weight updates. The scaler is also going to be fit only on the training data to prevent data leakage, not the entire dataset.

**How to detect overfitting when training**

We can detect overfitting when training the models by comparing the performance of training and holdout data. If the model performance on the training set keeps improving while the performance on the holdout only stays the same or gets worse, then the model is likely becoming overfit as it would create an increasing gap in performance between the two, showing that the model can't generalize to outside data.

# **All Imports** #

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.datasets import make_classification
from sklearn.preprocessing import OneHotEncoder

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [ ]:
# Data prep

X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size = 0.2, random_state = 100) # split

# fit scaler on training set and apply to holdout set (no leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32).reshape(-1, 1)

In [ ]:
# USE AS MANY CELLS AS NEEDED

# class

class BinaryMLP:
  # init statement
  def __init__(self, input_dim, hidden_dim, n_hidden_layers = 1, activation = 'relu', learning_rate = 0.01):
    self.n_hidden_layers = n_hidden_layers
    self.activation = F.relu if activation == 'relu' else F.tanh # set to relu if not changed, otherwise to tanh
    self.learning_rate = learning_rate

    # input layer
    self.W_1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
    self.b_1 = torch.randn(hidden_dim, requires_grad=True)
    self.params = [self.W_1, self.b_1]

    # hidden layer
    if n_hidden_layers == 2:
      self.W_hidden = torch.randn(hidden_dim, hidden_dim, requires_grad=True)
      self.b_hidden = torch.randn(hidden_dim, requires_grad=True)
      self.params += [self.W_hidden, self.b_hidden]

    # last hidden layer (to output layer with one unit)
    self.W_2 = torch.randn(hidden_dim, 1, requires_grad=True)
    self.b_2 = torch.randn(1, requires_grad=True)
    self.params += [self.W_2, self.b_2]

  # forward input
  def forward(self, X):
    # input to first hidden layer
    h = self.activation(torch.matmul(X, self.W_1) + self.b_1)
    # 2nd hidden layer if applicable
    if self.n_hidden_layers == 2:
      h = self.activation(torch.matmul(h, self.W_hidden) + self.b_hidden)
    # output layer with sigmoid to turn into probability
    out = torch.matmul(h, self.W_2) + self.b_2
    return torch.sigmoid(out)

  # cost function
  def cost(self, y_pred, y_true):
    prob = F.binary_cross_entropy(y_pred, y_true)
    return prob

  # fit function
  def fit(self, X, y, epochs = 100):
    for epoch in range(epochs):
      randperm = torch.randperm(X.shape[0])
      for i in randperm:
        x_batch = X[i]
        y_batch = y[i]

        # forward pass
        probs = self.forward(x_batch)
        cost = self.cost(probs, y_batch) # cost function

        # backward pass
        cost.backward()

        # update gradients and zero them
        with torch.no_grad():
          for param in self.params:
            param -= self.learning_rate * param.grad
            param.grad.zero_()
    return self

  def predict(self, X):
    with torch.no_grad():
      probs = self.forward(X)
      predictions = (probs > 0.5).float()
    return predictions

In [ ]:
# Model training and eval

configs = [
  (4, 1, 'relu'), (16, 1, 'relu'), (64, 1, 'relu'), # one layer, varying units
  (4, 2, 'relu'), (16, 2, 'relu'), (64, 2, 'relu'), # same units, two layers
  (4, 1, 'tanh'), (16, 1, 'tanh'), (64, 1, 'tanh'), # same stuff, but with tanh
  (4, 2, 'tanh'), (16, 2, 'tanh'), (64, 2, 'tanh'),
]

results = []
for hidden_dim, n_layers, act in configs:
  torch.manual_seed(123)
  model = BinaryMLP(input_dim = X_train.shape[1], hidden_dim = hidden_dim,
                    n_hidden_layers = n_layers, activation = act, learning_rate = 0.01)
  model.fit(X_train, y_train, epochs = 100)

  results.append({'activation': act,
    'hidden layers': n_layers,
    'units per layer': hidden_dim,
    'train accuracy': accuracy_score(y_train, model.predict(X_train)),
    'holdout accuracy': accuracy_score(y_holdout, model.predict(X_holdout))})

binary_results = pd.DataFrame(results)
binary_results

,activation,hidden layers,units per layer,train accuracy,holdout accuracy
0,relu,1,4,0.88500,0.890
1,relu,1,16,0.97375,0.985
2,relu,1,64,0.97625,0.985
3,relu,2,4,0.97750,0.975
4,relu,2,16,0.97000,0.985
5,relu,2,64,0.98125,0.995
6,tanh,1,4,0.97250,0.985
7,tanh,1,16,0.97500,0.985
8,tanh,1,64,0.97250,0.990
9,tanh,2,4,0.97875,0.980


- **How does the number of hidden units affect performance?**

With ReLU, increasing the number of hidden units increased performance (~89% acc with 4, to about 97-98% with 16 and 64). More units helped only until the model had enough to encapsulate the boundary (curved), so going beyond 16 units didn't further improve performance. With tanh, having just 4 units reached 97% train acc and 98.5% holdout acc, so the number of units mattered far less. None of the models extensively overfit since the train and holdout accuracies stayed quite close in every case. In fact, holdout accuracy was equal to or higher than train accuracy in almost every configuration, likely because of the noise in the data and the smaller holdout set. Increasing the number of units much further could eventually lead to overfitting.

- **What changes when using two layers instead of one?**

Adding a 2nd layer had the largest effect when the model had fewer units (4). With ReLU and 4 units, accuracy jumped up from 88.5%/89% with 1 layer to 97.75%/97.5% accuracy with 2 layers since stacking ReLU layers lets the network combine the linear pieces and form a more curved boundary that we're looking for. Once there were 16 or more hidden units, just one layer was already flexible enough for this dataset, so the additional layer changed accuracy very little, adding complexity for little to no payoff. For tanh, adding the 2nd layer slightly increased train accuracy while holdout accuracy slightly decreased, which could be a very slight sign of the model fitting on the noise. Regardless of this, on a 200 sample holdout, a 1% difference is only 2 predictions, so the difference isn't super meaningful or impactful. The best holdout accuracy overall was with ReLU with 2 layers and 64 hidden units with an impressive holdout accuracy of 99.5%.

- **How does the activation function (ReLU vs. Tanh) influence results?**

The biggest difference was when using 4 hidden units in a single hidden layer. Tanh already got to 97% accuracy with 4 units while ReLU was only at 89% or so, showing a clear difference between them. A potential reason for this is that ReLU outputs zero for negative inputs, so having fewer units means some of them can just not activate and the model loses some of the capacity for this. Tanh, on the other hand, is a smooth curved function, allowing each unit to influence and bend the decision boundary more than a ReLU unit would be able to. These ReLU units only create linear pieces, not curved. Since each configuration was trained with just one random seed, part of the low ReLU result could also be because of unlucky initialization. More specifically, if a unit's random weights and bias make the inputs negative for every data point, it will just output 0 every time, receiving no gradient and not updating at all. With only 4 units, losing any of them leaves a lot less capacity for the model to fit the curved boundary. With 16+ hidden units, both activation functions performed about the same as mentioned before (about 97% train acc, 98-99% holdout acc) since there were plenty of units to make up for the restrictions/limitations that ReLU causes/has. Overall, the difference was only really visible at 4 hidden units with 1 hidden layer, with tanh performing better than ReLU at said amount. With 2 hidden layers and 4 units, ReLU (97.75%) and tanh (97.875%) performed essentially the same since the additional layer made up for the limitations ReLU set. Most of the differences in the table are only 0.5-1% on a 200 sample holdout set, which is only 1 or 2 predictions of change, so this can easily fall within the range of accounting for just random noise.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$-\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3,
                           n_clusters_per_class=1,
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
# split the data
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, random_state=42)

# fit on training set, then transform both
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

# one-hot encode the labels
encoder = OneHotEncoder(sparse_output = False)
y_train = encoder.fit_transform(y_train.reshape(-1, 1))
y_holdout = encoder.transform(y_holdout.reshape(-1, 1))

# convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32)

# implementation
class MultiClassMLP(object):
  def __init__(self, input_dim, hidden_dim, output_dim, n_hidden_layers=1, activation='relu', learning_rate=0.01):
    self.n_hidden_layers = n_hidden_layers
    self.learning_rate = learning_rate
    self.activation = torch.relu if activation == 'relu' else torch.tanh

    # first hidden layer
    self.W_1 = (torch.randn(input_dim, hidden_dim) * 0.1).requires_grad_()
    self.b_1 = torch.zeros(1, hidden_dim, requires_grad=True)
    self.params = [self.W_1, self.b_1]

    # optional second hidden layer
    if n_hidden_layers == 2:
      self.W_h = (torch.randn(hidden_dim, hidden_dim) * 0.1).requires_grad_()
      self.b_h = torch.zeros(1, hidden_dim, requires_grad=True)
      self.params += [self.W_h, self.b_h]

    # output layer
    self.W_2 = (torch.randn(hidden_dim, output_dim) * 0.1).requires_grad_()
    self.b_2 = torch.zeros(1, output_dim, requires_grad=True)
    self.params += [self.W_2, self.b_2]

  def forward(self, X):
    h = self.activation(X @ self.W_1 + self.b_1)
    if self.n_hidden_layers == 2:
      h = self.activation(h @ self.W_h + self.b_h)
    logits = h @ self.W_2 + self.b_2
    # return probabilities
    return F.softmax(logits, dim=1)

  # categorical cross-entropy loss
  def cost(self, y_pred, y_true):
    y_pred = torch.clamp(y_pred, 1e-12, 1.0)   # avoid log(0)
    return -torch.mean(torch.sum(y_true * torch.log(y_pred), dim=1))

  # fit the model
  def fit(self, X, Y, epochs=150):
    n = X.shape[0]
    for epoch in range(epochs):
      perm = torch.randperm(n)
      for i in perm:
        # stochastic gradient descent, one observation per update
        x_i = X[i:i + 1]
        y_i = Y[i:i + 1]
        y_pred = self.forward(x_i)
        loss = self.cost(y_pred, y_i)
        loss.backward()
        with torch.no_grad():
          for param in self.params:
            param -= self.learning_rate * param.grad
            param.grad.zero_()
    return self

  # predict
  def predict(self, X):
    with torch.no_grad():
      # class with highest probability
      return self.forward(X).argmax(dim=1)

In [ ]:
# model training and evaluation
def accuracy(model, X, Y):
    return (model.predict(X) == Y.argmax(dim=1)).float().mean().item()

results = []
for activation in ['relu', 'tanh']:
    for hidden_layers, hidden_units in [(1, 4), (1, 8), (1, 16), (2, 8), (2, 16)]:
        torch.manual_seed(1234)
        model = MultiClassMLP(input_dim=2, hidden_dim=hidden_units, output_dim=3,
                              n_hidden_layers=hidden_layers, activation=activation,
                              learning_rate=0.01)
        model.fit(X_train, y_train)
        results.append({
            'activation': activation,
            'hidden_layers': hidden_layers,
            'hidden_units': hidden_units,
            'train_accuracy': accuracy(model, X_train, y_train),
            'holdout_accuracy': accuracy(model, X_holdout, y_holdout),
        })

multiclass_results = pd.DataFrame(results)
multiclass_results

,activation,hidden_layers,hidden_units,train_accuracy,holdout_accuracy
0,relu,1,4,0.81250,0.795
1,relu,1,8,0.81625,0.790
2,relu,1,16,0.81625,0.795
3,relu,2,8,0.82375,0.795
4,relu,2,16,0.82125,0.790
5,tanh,1,4,0.81750,0.795
6,tanh,1,8,0.81625,0.795
7,tanh,1,16,0.82000,0.795
8,tanh,2,8,0.81625,0.800
9,tanh,2,16,0.82125,0.795


4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?

For the ReLU activation function and one hidden layer, increasing the number of hidden units from 4 to 8 increased the training accuracy by 0.00375, but stayed the same when increasing the number of hidden units from 8 to 16. The holdout accuracy fluctuated by 0.005 when increasing from 4 to 8 to 16 hidden units. For ReLU with two hidden layers, the training accuracy and holdout accuracy decreased slightly as the number of hidden units increased. However, these differences are marginal and show that 4 units is enough, and it is redundant to increase the number of hidden units to be higher. The data set is small and simple, so a small network of 4 hidden units can learn the decision boundaries well.

- What changes when using two layers instead of one?

When using two hidden layers instead of one, the training accuracy is higher for the highest number of hidden units (16) compared to the configurations with just one hidden layer and 16 hidden units. The configuration with the highest holdout accuracy is tanh with two hidden layers and eight hidden units, with a holdout accuracy of 0.800. Like with the hidden units, though, changing the number of hidden layers produces marginal differences in results. Adding a second layer adds capacity that this problem does not need because the data set is not very complex; it only has two features with three clusters.

- How does the activation function (ReLU vs. Tanh) influence results?

The results between ReLU and Tanh are quite similar, but the Tanh holdout accuracy is higher on average compared to ReLU across the different configurations (average of 0.796 vs 0.793). This is a slight difference of only 0.003, though, so it also demonstrates how either activation function is sufficient for this problem and can detect non-linear boundaries. Additionally, since the holdout set has 200 points, one point equals 0.005 accuracy. A difference this small is less than even one prediction on the holdout, so it is within the range of random noise. The noise also explains why every configuration seems to flatten out/plateau around 80% or so since flip_y is 15% or 0.15, randomly assigning 15% of the labels, so a portion of these points can't really be predicted correctly by any model regardless of the chosen activation function or structure and such.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED
# Point 1: Implementation of class

class RegressionMLP(torch.nn.Module):
  def __init__(self, input_size, hidden_size, output_size=1, hidden_size2=None, activation='relu'):
    super().__init__()
    self.activation = activation
    self.hidden_size2 = hidden_size2
    # hidden layer weights and bias
    self.fc1 = torch.nn.Linear(input_size, hidden_size)

    # second hidden layer
    if hidden_size2 is not None:
      self.fc2 = torch.nn.Linear(hidden_size, hidden_size2)
      self.fc_out = torch.nn.Linear(hidden_size2, output_size)
    else:
      self.fc_out = torch.nn.Linear(hidden_size, output_size)

    for layer in self.modules():
      if isinstance(layer, torch.nn.Linear):
        torch.nn.init.normal_(layer.weight, mean=0.0, std=0.01)
        torch.nn.init.zeros_(layer.bias)

  def _activate(self, z):
    if self.activation == 'relu':
      return F.relu(z)
    return torch.tanh(z)

  def forward(self, X):
    a1 = self._activate(self.fc1(X))
    if self.hidden_size2 is not None:
      a2 = self._activate(self.fc2(a1))
      return self.fc_out(a2)
    return self.fc_out(a1)

  def cost(self, y_pred, y_true):
    # MSE loss
    return F.mse_loss(y_pred, y_true)

  def fit(self, X, y, learning_rate=0.01, epochs=1000):
    # will accept numpy arrays, pandas objects, or tensors
    X = torch.as_tensor(np.asarray(X), dtype=torch.float32)
    y = torch.as_tensor(np.asarray(y), dtype=torch.float32).reshape(-1, 1)
    n = X.shape[0]

    for epoch in range(epochs):
      # SGD: iterate over each sample with new shuffled order for each epoch
      for i in torch.randperm(n):
        xi = X[i:i+1] # shape (1, features)
        yi = y[i:i+1] # shape (1, 1)

        loss = self.cost(self.forward(xi), yi)
        loss.backward()

        with torch.no_grad():
          for param in self.parameters():
            param -= learning_rate * param.grad
            param.grad.zero_()
    return self

  def predict(self, X):
    X = torch.as_tensor(np.asarray(X), dtype=torch.float32)
    with torch.no_grad():
      return self.forward(X)

In [ ]:
# Point 2: Data Preparation

X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size = 0.2, random_state = 100) # split

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_holdout = y_scaler.transform(y_holdout.reshape(-1, 1))

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32)

In [ ]:
# Point 3: Model Training and Evaluation

X_train_np = X_train.numpy()
y_train_np = y_train.numpy()
X_holdout_np = X_holdout.numpy()
y_holdout_np = y_holdout.numpy()

configs = [
    # (hidden_size, hidden_size2, activation)
    (8, None, 'relu'),
    (16, None, 'relu'),
    (32, None, 'relu'),
    (16, 16, 'relu'),
    (32, 32, 'relu'),
    (8, None, 'tanh'),
    (16, None, 'tanh'),
    (32, None, 'tanh'),
    (16, 16, 'tanh'),
    (32, 32, 'tanh'),
]

# true values in original units
y_train_orig = y_scaler.inverse_transform(y_train_np)
y_holdout_orig = y_scaler.inverse_transform(y_holdout_np)

results = []
for h1, h2, act in configs:
  torch.manual_seed(123)
  model = RegressionMLP(input_size=2, hidden_size=h1, hidden_size2=h2, activation=act)
  model.fit(X_train_np, y_train_np, learning_rate=0.01, epochs=100)

  train_pred = y_scaler.inverse_transform(model.predict(X_train_np).numpy())
  holdout_pred = y_scaler.inverse_transform(model.predict(X_holdout_np).numpy())
  train_mse = mean_squared_error(y_train_orig, train_pred)
  holdout_mse = mean_squared_error(y_holdout_orig, holdout_pred)
  results.append({
      'Hidden Layers': 2 if h2 else 1,
      'Hidden Units': f"{h1}-{h2}" if h2 else str(h1),
      'Activation': act,
      'Train MSE': round(train_mse, 2),
      'Holdout MSE': round(holdout_mse, 2)
  })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 Hidden Layers Hidden Units Activation  Train MSE  Holdout MSE
             1            8       relu    5472.78      6222.69
             1           16       relu    5383.65      5773.64
             1           32       relu    5518.30      5913.08
             2        16-16       relu    5467.15      5829.17
             2        32-32       relu    5455.46      5852.34
             1            8       tanh    5531.44      6148.02
             1           16       tanh    5516.29      5975.92
             1           32       tanh    5478.67      6089.56
             2        16-16       tanh    5670.92      6095.73
             2        32-32       tanh    5487.34      5903.87


### Point 4: Reflection and Discussion

* How does the number of hidden units affect performance?

In general, the number of hidden units did not have a large impact on performance. With 1 hidden layer, 8 units gave the highest holdout MSE for both ReLU (6222.69) and tanh (6148.02), and 16 units gave the lowest (5773.64 and 5975.92). While using 16 units is slightly better, using 32 neurons appears to have a minimal difference in performance. However, every model's training MSE is already close to 5625 (75^2), which is the variance of the noise added by make_regression, which is the lowest MSE a model can realistically reach. This means that none of the models underfit and the differences between unit counts are pretty much mainly random noise. Holdout MSE was higher than the training MSE in every configuration by about 350-750, so the models may fit a small amount of the training noise, but this gap doesn't grow with more units so there's no real strong sign of overfitting either, which is good.

* What changes when using two layers instead of one?

Across all values of neurons, using two layers instead of one did not have a noticeable impact on MSE. There was no consistent pattern: with ReLU and 16 units, the 2nd layer slightly raised both training and holdout MSE, while with ReLU and 32 units it slightly lowered both. The relationship generated by make_regression is linear, so one hidden layer already has more than enough capacity and the extra layer has nothing else to actually learn. The differences between the values are so small that there is not enough evidence to suggest that the model will overfit with an extra layer.

* How does the activation function (ReLU vs. Tanh) influence results?

When referencing the results above, using the ReLU activation function leads to slightly better results compared to the Tanh function. ReLU had a lower holdout MSE in 4/5 configurations (avg 5920 vs 6043) as well as a lower average training MSE (5460 vs 5537). One possible explanation is that make_regression creates a linear relationship between the features and the target. ReLU is a piecewise linear function, so it can represent linear trends quite well and easily, while tanh is curved and has to approximate a straight line. However, the difference between the two is so small (about 120 in holdout MSE or 2%) that using either one won't have a severe impact on the actual results.



## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

One way that we could have leveraged inheritance in our code could have been to make an MLP class with the functions that each of our classes shared: `__init__`, forward, and fit. The base class would hold __init__ for creating the hidden layer weights and biases (1 or 2 layers, ReLU or tanh), the hidden layer forward pass, and the SGD fit loop since that loop is the same for every task. This would include forward pass, computing the cost, backward pass, and manually updating each parameter. This way our BinaryMLP, MultiClassMLP, and RegressionMLP classes could be more concise and only include the functions unique to the specified mode of processing (output activation, cost, and predict). For example, BinaryMLP could inherit from the base class and only define a sigmoid output with 1 unit, binary cross-entropy for the cost function, and predict with a threshold of 0.5. MultiClassMLP would define a softmax output with 3 units, categorical cross-entropy, and predict using the argmax. Finally, RegressionMLP would use a linear output with 1 unit, MSE, and predict using raw output. Each subclass would use `super().__init__()` to build the hidden layers they share together, so a fix to the fit loop or hidden layers would only need to be made one time rather than every single time individually for each class.

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE

While we did not encounter any major problems, one aspect of our collaboration that could still be improved relates to the previous discussion question about preventing duplication. Given that we preprocessed our data before each class, we could have also created a data preprocessor class instead of running the same logic three separate times to avoid duplication in our early steps. This would have been a much more straightforward solution compared to what we did, so a way we can avoid this next time is to check for similarities between sections and potentially create a parent function that each of us can use in our own sections without influencing the other's code.